GET DETAILS ABOUT CATALOG AND SCHEMA

In [0]:
%sql
SELECT current_catalog(), current_schema()

In [0]:
%sql
LIST '/Workspace/Users/lcalbente@gmail.com/databricks-projects/'

Create Schema and Volume for Spotify projects

In [0]:
%sql
-- CREATE SCHEMA IF NOT EXISTS workspace.databricks_projects;

-- CREATE VOLUME IF NOT EXISTS workspace.databricks_projects.spotify;
-- CREATE VOLUME IF NOT EXISTS workspace.databricks_projects.jobs;
CREATE VOLUME IF NOT EXISTS workspace.databricks_projects.financial_data;

AFTER IMPORT CSV FILE TO DATABRICKS WORKSPACE
COPY DATASET TO databrics_projects VOLUME

In [0]:
dbutils.fs.mkdirs("/Volumes/workspace/databricks_projects/spotify")
dbutils.fs.mkdirs("/Volumes/workspace/databricks_projects/jobs")

dbutils.fs.cp(
  "/Volumes/workspace/databricks_projects/spotify/dataset.csv",
  "/Volumes/workspace/databricks_projects/spotify"
)
dbutils.fs.cp(
  "/Volumes/workspace/databricks_projects/jobs/DataEngineer.csv",
  "/Volumes/workspace/databricks_projects/jobs"
)

READ CSV FILE USING SQL

In [0]:
%sql
SELECT *
FROM read_files(
    '/Volumes/workspace/databricks_projects/jobs',
    format => 'csv',
    header => 'true',
    inferSchema => 'true',
    multiline => 'true'
  )

CREATE TABLE WITH SQL STATEMENT AND CSV FILES

In [0]:
%sql

-- DROP THE TABLE IF IT EXISTS
-- DROP TABLE IF EXISTS data_engineer_jobs;

-- -- CREATE DELTA TABLE
-- CREATE TABLE data_engineer_jobs
-- SELECT 
--     `Job Title` as job_title,
--     `Salary Estimate` as salary_estimate,
--     `Job Description` as job_description,
--     `Rating` as rating,
--     `Company Name` as company_name,
--     `Location` as Location,
--     `Headquarters` as headquarters,
--     `Size` as size,
--     `Founded` as founded,
--     `Type of ownership` as type_of_ownership,
--     `Industry` as industry,
--     `Sector` as sector,
--     `Revenue` as revenue,
--     `Competitors` as competitors,
--     `Easy Apply` as easy_apply
-- FROM read_files(
--     '/Volumes/workspace/databricks_projects/jobs',
--     format => 'csv',
--     header => 'true',
--     inferSchema => 'true',
--     multiline => 'true'
--     );

-- PREVIEW DELTA TABLE
SELECT *
FROM data_engineer_jobs
LIMIT 10;

GET TABLE INFORMATION

In [0]:
%sql
DESCRIBE TABLE EXTENDED data_engineer_jobs

In [0]:
READ CSV FILE WITH PYSPARK

In [0]:
import re

def clean_column_name(name):
    clean_name = name.lower().replace(" ", "_").replace("(", "").replace(")", "")
    clean_name = re.sub(r"[^a-zA-Z0-9_]", "", clean_name)
    return clean_name

# Read CSV file from volume into a Spark DataFrame
datasets_df = (spark.read
               .format("csv")
               .option("header", "true")
               .option("inferSchema", "true")
               .option("multiline","true")
               .option("rescuedDataColumn", "_rescued_data")  # Add column for rescued data
               .load("/Volumes/workspace/databricks_projects/jobs/DataEngineer.csv")
               .select("*", "_metadata")  # Add metadata column
            )

new_columns = [clean_column_name(col) for col in datasets_df.columns]
df_cleaned = datasets_df.toDF(*new_columns)

display(df_cleaned)


In [0]:
# Write to the DataFrame to a Delta table (overwrite if it exists, can also use other types of write modes)
# (df_cleaned
#  .write
#  .mode("overwrite")
#  .saveAsTable("databricks_projects.data_engineer_jobs")
# )

# ## Read and view the table
datasets_table = spark.table("databricks_projects.data_engineer_jobs")


In [0]:
%sql
DESCRIBE EXTENDED databricks_projects.data_engineer_jobs

In [0]:
READ CSV WITH PYTHON

In [0]:
import pandas as pd

datasets_df = pd.read_csv(
    "/Volumes/workspace/databricks_projects/jobs/DataEngineer.csv",
    header='infer',
    select=('_metadata', '*')
    )   

print(datasets_df.columns)
print(datasets_df.head())

INCREMENTAL DATA INGESTION USING COPY INTO
  legacy function, recommended using read_files()

In [0]:
%sql
-- First example: Create empty table with wrong schema and fix the error

DROP TABLE IF EXISTS spotify_dataset;

CREATE TABLE spotify_dataset(
  track_id STRING,
  artists STRING,
  popularity STRING
);

COPY INTO spotify_dataset
FROM '/Volumes/workspace/spotify/databricks_projects/dataset.csv'
FILEFORMAT = CSV;

In [0]:
%sql
-- Fixing the error, we can use mergeSchema = 'true', this options allows the incoming data schema being merged with the table schema

DROP TABLE IF EXISTS spotify_dataset;

CREATE TABLE spotify_dataset(
  track_id STRING,
  artists STRING,
  popularity STRING
);

COPY INTO spotify_dataset
FROM '/Volumes/workspace/spotify/databricks_projects/dataset.csv'
FILEFORMAT = CSV
COPY_OPTIONS ('mergeSchema' = 'true');


In [0]:
%sql
-- Second Example: Preemptively Handling Schema Evolution
-- 1. create an empty table named spotify_datasets_no_schema
-- 2. Add COPY_OPTIONS ('mergeSchema' = 'true')

DROP TABLE IF EXISTS spotify_dataset_no_schema;

CREATE TABLE spotify_dataset_no_schema;

COPY INTO spotify_dataset_no_schema
FROM '/Volumes/workspace/spotify/databricks_projects/dataset.csv'
FILEFORMAT = CSV
COPY_OPTIONS ('mergeSchema' = 'true');


IDEMPOTENCY (INCREMENTAL INGESTION)

In [0]:
%sql
COPY INTO spotify_dataset_no_schema
FROM '/Volumes/workspace/spotify/databricks_projects/dataset.csv'
FILEFORMAT = CSV
COPY_OPTIONS ('mergeSchema' = 'true')

In [0]:
### UTILS


### 1. See files structure
spark.sql(f'''
  SELECT *
  FROM text.`/Volumes/workspace/spotify/databricks_projects/dataset.csv` 
''').display()